# CellGPS spatial topology from AnnData

This notebook computes CellGPS/COSTE Searcher and Findee StructureMaps from segmented cell centroids and a cell-group annotation. The input must be a **spatial** AnnData object: a conventional scRNA-seq reference without cell coordinates is not sufficient.

Keep this notebook on the `ISS_postprocessing` kernel. Create the isolated CPU-only CellGPS runtime once from the repository root:

```bash
conda env create -f ISS_postprocessing/environments/cellgps.yml
```

No GPU is required.

In [ ]:
from pathlib import Path
import json

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import ISS_postprocessing.spatial_topology as TOPOLOGY

## Select the spatial object and annotation

The single-cell ISS tutorial creates `obsm['spatial']` and Leiden columns such as `obs['norm_leiden_2']`. Replace these paths and names with the values in your object. If several tissue sections were concatenated in one object, consider running each section separately so distances are never measured across unrelated coordinate systems.

In [ ]:
h5ad_path = Path('/path/to/All regions/postprocessing/segmentation/anndata_cellpose.h5ad')
region = 'R1'
cluster_col = 'norm_leiden_2'
coordinate_key = 'spatial'
coordinate_unit = 'pixel'  # use 'um' if obsm[coordinate_key] is in micrometres
output_dir = h5ad_path.parent / 'spatial_topology' / 'cellgps' / 'cells'

adata = ad.read_h5ad(h5ad_path)
adata

In [ ]:
if cluster_col not in adata.obs:
    raise KeyError(f'{cluster_col!r} is not in adata.obs')
if coordinate_key not in adata.obsm:
    raise KeyError(f'{coordinate_key!r} is not in adata.obsm')

display(adata.obs[cluster_col].value_counts(dropna=False).rename('n_cells').to_frame())
print('Spatial array shape:', adata.obsm[coordinate_key].shape)

## Validate and prepare without running CellGPS

Preparation extracts only x/y coordinates and group labels, omits unlabelled cells, filters very small groups, and writes a reproducible configuration. It does not alter `adata` and does not require CellGPS.

In [ ]:
parameters = dict(
    min_cells_per_group=5,
    linkage_method='average',
    batch_size=100_000,
    show_corr=True,
)

prepared = TOPOLOGY.prepare_cellgps_from_h5ad(
    adata,
    region=region,
    cluster_col=cluster_col,
    coordinate_key=coordinate_key,
    coordinate_unit=coordinate_unit,
    output_dir=output_dir,
    **parameters,
)
points = pd.read_csv(prepared['points'])
display(points.head())
display(points['celltype'].value_counts().rename('n_cells').to_frame())
print(json.dumps(json.loads(prepared['config'].read_text()), indent=2))

## Run CellGPS/COSTE

The wrapper launches CellGPS 0.0.6 through its dedicated Conda environment and uses the memory-optimized dataframe implementation. Set `overwrite=True` after changing parameters; otherwise a completed result is reused.

In [ ]:
outputs = TOPOLOGY.run_cellgps_from_h5ad(
    adata,
    region=region,
    cluster_col=cluster_col,
    coordinate_key=coordinate_key,
    coordinate_unit=coordinate_unit,
    output_dir=output_dir,
    cellgps_python=['conda', 'run', '-n', 'cellgps', 'python'],
    overwrite=False,
    **parameters,
)
outputs

## Inspect the StructureMaps

Rows are Searcher entities and columns are Findee entities in the directed nearest-neighbour calculation. CellGPS clusters the row and column profiles separately, so the two normalized cophenetic matrices need not be identical. Values summarize topology between annotated cell groups; they are not expression fold changes or physical distances.

In [ ]:
searcher = pd.read_csv(outputs['searcher_matrix'], index_col=0)
findee = pd.read_csv(outputs['findee_matrix'], index_col=0)
metadata = json.loads(outputs['metadata'].read_text())

display(pd.read_csv(outputs['entity_counts']))
print(json.dumps(metadata, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(searcher, cmap='RdBu', vmin=0, vmax=1, square=True, ax=axes[0])
axes[0].set_title('Searcher StructureMap')
sns.heatmap(findee, cmap='RdBu', vmin=0, vmax=1, square=True, ax=axes[1])
axes[1].set_title('Findee StructureMap')
plt.tight_layout();

## Recommended checks

- Run tissue sections separately unless their coordinates share a genuine common frame.
- Repeat the analysis with a higher `min_cells_per_group` to check that rare groups do not drive the hierarchy.
- Compare biologically meaningful annotations with unsupervised clusters, but do not interpret cluster-number ordering itself.
- Use the saved PDF files for the native CellGPS clustered/dendrogram view.